# Applied Benchmarks — Resume Parsing & Resume Writing

Scope: **our own measured end-to-end runs**, not AA lab scores and not router spend.
Costs here are real USD from those runs (`SRC_BENCH` provenance).

- V2 Enhanced resume **parsing** — 18 models × 5 resumes, 110-point scoring
- V3 resume **writing** — 17 models writing for a Boeing Defense GenAI DS role,
  100-point scoring, fact-fidelity audit

Model-lookup tables are in `AA_Model_Efficiency_Leaderboard.ipynb`; real spend is in
`OpenRouter_Spend_Analytics.ipynb`.


In [ ]:
from dotenv import load_dotenv
load_dotenv()  # reads .env (keys are never hardcoded)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import os
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

# API Configuration
import os
API_KEY = os.getenv("ARTIFICIAL_ANALYSIS_KEY")
BASE_URL = "https://artificialanalysis.ai/api/v2/data/llms/models"

# ── Data-provenance stamps: every figure gets a source note (bottom-left) ──
SRC_AA = ('Source: Artificial Analysis API (artificialanalysis.ai/api/v2) — lab '
          'intelligence index, IFBench reliability, listed blended 3:1 price.')
SRC_AA_OR = ('Source: AA API (intel, reliability, listed price); OpenRouter price '
             'fallback for models AA lists at $0.')
SRC_OR_TEL = ('Source: OpenRouter endpoints API — measured throughput_last_30m.p50 '
              '(rolling 30-min real traffic) + per-endpoint blended price; gates: '
              'uptime>=95%, ctx>=100k, no fp4/mxfp4 quant.')
SRC_BENCH = ('Source: our measured benchmark runs (v3_resume_writing_results.json, '
             'v2_benchmark_results.json) — real wall-clock latency, tokens, USD cost.')

def note_source(text=SRC_AA, fig=None, x=0.01, y=0.005):
    """Stamp a data-provenance note onto the current (or given) figure."""
    f = fig if fig is not None else plt.gcf()
    f.text(x, y, text, fontsize=7, color='#666666',
           ha='left', va='bottom', style='italic')

# ── Reasoning-variant labeling ──────────────────────────────────────────────
# AA lists reasoning-effort variants explicitly: (Max)/(Xhigh)/(High)/(Medium)/
# (Low)/(Reasoning)/(Non-reasoning)/("X, Default Fallback"). Rankings compare
# specific variants, so every plot label keeps its reasoning level visible.
import re as _re_lv
_VARIANT_RE = _re_lv.compile(r'\(([^)]+)\)\s*$')

def variant_of(name):
    """Extract the reasoning-effort level from an AA model name ('' if none)."""
    m = _VARIANT_RE.search(str(name or ''))
    if not m:
        return ''
    v = m.group(1)
    v = _re_lv.sub(r'\s*,?\s*(Default Fallback|Opus 4\.8 Fallback)\s*', '', v)
    v = _re_lv.sub(r'\s*,\s*', ' + ', v)
    return v.strip()

def fam_label(name, maxlen=44):
    """'Family (LEVEL)' label — keeps the reasoning level visible even when truncated."""
    s = str(name or '')
    v = variant_of(s)
    if v:
        s = _VARIANT_RE.sub('', s).strip()
        return (s[:maxlen - 2 - len(v)] + ' (' + v + ')') if len(s) + len(v) + 3 > maxlen else s + ' (' + v + ')'
    return s[:maxlen]


## Appendix

Additional benchmarks for resume parsing and resume writing tasks using the same LLM models.

In [ ]:
# ============================================================================
# CONSOLIDATED: V2 Enhanced Resume Parsing Benchmark
# 18 Models × 5 Resumes (3 domains) — 110-point scoring with name & date accuracy
# ============================================================================

import json, pandas as pd, matplotlib.pyplot as plt, numpy as np

# Try to load V2 results, fall back to V1 leaderboard
try:
    with open('/home/skystarved/dev_workspaces/results/v2_benchmark_results.json', 'r') as f:
        v2 = json.load(f)
    data = []
    for e in v2:
        data.append({
            'model': e['model'], 'score': e['avg_score'], 'pct': e['avg_pct'],
            'tokens': e['total_tokens'], 'time_5': e['total_time'],
            'time_per': e['time_per_resume'], 'price_per_M': e['price_blended'],
            'cost': e['cost'], 'cost_per': e['cost'] / 5,
            'reliability': e['reliability'], 'scored': True,
        })
    bench_df = pd.DataFrame(data)
    dfs = bench_df[bench_df['scored']]
    scoring = 'V2 Enhanced (110 pts): Schema(20) + Name(5) + Title(10) + Exp(15+5) + Dates(5) + Edu(10) + Certs(10) + Proj(10) + Skills(10) + Bullets(5) + Domain(5) + Vol/Aff(5)'
except:
    # Fallback to V1 leaderboard
    with open('/home/skystarved/dev_workspaces/results/final_complete_leaderboard.json', 'r') as f:
        lb = json.load(f)
    data = []
    for e in lb['complete_leaderboard']:
        tokens = e.get('tokens_5_resumes'); time_5 = e.get('time_5_resumes')
        if tokens is None or time_5 is None or time_5 == 0: continue
        price = float(e.get('price_blended_per_M', '$0').replace('$', ''))
        cost = (tokens / 1_000_000) * price
        score = e.get('avg_score', 0)
        scored = isinstance(score, (int, float)) and score > 0
        data.append({'model': e['model'], 'score': score if scored else 0,
            'tokens': tokens, 'time_5': time_5, 'time_per': time_5 / 5,
            'price_per_M': price, 'cost': cost, 'cost_per': cost / 5,
            'reliability': e.get('reliability', '?'), 'scored': scored})
    bench_df = pd.DataFrame(data); dfs = bench_df[bench_df['scored']]
    scoring = 'V1 (100 pts): Schema(20) + Title(10) + Exp(15+5) + Edu(10) + Certs(10) + Proj(10) + Skills(10) + Bullets(5) + Domain(5)'

print("=" * 85)
print("CONSOLIDATED RESUME PARSING BENCHMARK — 18 Models")
print("=" * 85)
print(f"Models scored: {len(dfs)} | {scoring}")
print(f"Total tokens: {bench_df['tokens'].sum():,} | Total cost: ${bench_df['cost'].sum():.2f}")

# Leaderboard table
print(f"\n{'#':<3} {'Model':<26} {'Score':<10} {'%':<6} {'Time':<8} {'$/Res':<10} {'Price/M':<9} {'Reliab'}")
print("-" * 90)
for i, (_, r) in enumerate(dfs.sort_values('score', ascending=False).iterrows(), 1):
    if r['score'] > 100:  # V2 110-point scale
        sc = f"{r['score']:.1f}/110"; pct = f"{r['score']/110*100:.1f}%"
    else: sc = f"{r['score']:.1f}%"; pct = ""
    print(f"{i:<3} {r['model']:<26} {sc:<10} {pct:<6} {r['time_5']:.0f}s{'':<3} ${r['cost_per']:<9.5f} ${r['price_per_M']:<7.3f}/M {r['reliability']}")

# Consolidated 4-plot dashboard
# Use a smaller figure size and let matplotlib manage layout to avoid
# VS Code rendering issues where large figures end up mostly blank.
fig, axes = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True)

# 1) Speed
ax = axes[0, 0]; d = dfs.sort_values('time_5')
bars = ax.barh(d['model'], d['time_5'], color=plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(d)))[::-1])
ax.set_xlabel('Time for 5 Resumes (s)'); ax.set_title('Speed to Complete')
for b, v in zip(bars, d['time_5']): ax.text(v + 2, b.get_y() + b.get_height()/2, f'{v:.0f}s', va='center', fontsize=8)

# 2) Score
ax = axes[0, 1]; d = dfs.sort_values('score')
bars = ax.barh(d['model'], d['score'], color=plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(d))))
# Auto-scale xlim
max_score = d['score'].max(); ax.set_xlim(max_score * 0.95, max_score * 1.02)
ax.set_xlabel('Score'); ax.set_title('Parsing Accuracy')
for b, s in zip(bars, d['score']): ax.text(s + 0.1, b.get_y() + b.get_height()/2, f'{s:.1f}', va='center', fontsize=8)

# 3) Cost
ax = axes[1, 0]; d = dfs.sort_values('cost')
bars = ax.barh(d['model'], d['cost'], color=plt.cm.viridis(np.linspace(0.2, 0.9, len(d))))
ax.set_xlabel('Cost ($) for 5 Resumes'); ax.set_title('Total Cost')
for b, c in zip(bars, d['cost']): ax.text(c + 0.0002, b.get_y() + b.get_height()/2, f'${c:.5f}', va='center', fontsize=8)

# 4) Speed vs Score bubble
ax = axes[1, 1]
sc = ax.scatter(dfs['time_5'], dfs['score'], s=dfs['price_per_M'] * 2000, alpha=0.7,
                c=dfs['score'], cmap='RdYlGn', edgecolors='black', linewidth=0.5)
ax.set_xlabel('Time (5 resumes) → faster'); ax.set_ylabel('Score → higher'); ax.set_title('Speed × Score (bubble = price)')
ax.grid(alpha=0.3)
# Annotate only the top subset to avoid layout explosion / overlapping text
dfs_annot = dfs.sort_values('score', ascending=False).head(12)
for _, r in dfs_annot.iterrows():
    ax.annotate(
        str(r['model'])[:16],
        (r['time_5'], r['score']),
        xytext=(3, 2),
        textcoords='offset points',
        fontsize=7
    )
ax.axhline(dfs['score'].median(), color='gray', ls='--', alpha=0.3)
ax.axvline(dfs['time_5'].median(), color='gray', ls='--', alpha=0.3)
ax.text(0, dfs['score'].max() + 0.3, '★ FAST+SMART', fontweight='bold', color='green', fontsize=10)

plt.suptitle('Resume Parsing Benchmark — V2 Enhanced (Name, Dates, 3 Domains) — 18 Models', fontsize=14, fontweight='bold', y=1.01)
note_source(SRC_BENCH)
plt.show()

# Recommendations
print("\n" + "=" * 70)
print("RECOMMENDATIONS")
print("=" * 70)
best = dfs.loc[dfs['score'].idxmax()]; fast = dfs.loc[dfs['time_5'].idxmin()]; cheap = dfs.loc[dfs['cost'].idxmin()]
dfs['value'] = dfs['score'] / dfs['cost']; best_val = dfs.loc[dfs['value'].idxmax()]
print(f"  🏆 Best Score:   {best['model']} ({best['score']:.1f})")
print(f"  ⚡ Fastest:      {fast['model']} ({fast['time_per']:.1f}s/resume)")
print(f"  💰 Cheapest:     {cheap['model']} (${cheap['cost']:.4f} total)")
print(f"  📊 Best Value:   {best_val['model']} ({best_val['value']:.0f} score/dollar)")
# Dynamic best balance: highest score among models with time_per < median
median_time = dfs['time_per'].median()
fast_smart = dfs[dfs['time_per'] <= median_time].sort_values('score', ascending=False)
if not fast_smart.empty:
    bal = fast_smart.iloc[0]
    print(f"  🏅 Best Balance: {bal['model']} — {bal['score']:.1f}/110 at {bal['time_per']:.1f}s/resume, ${bal['price_per_M']:.2f}/M")

print("\nSpeed ranking:")
for _, r in dfs.sort_values('time_per').iterrows():
    print(f"  {r['model']:<28} {r['time_per']:.1f}s  |  {r['score']:.1f}  |  ${r['price_per_M']:.2f}/M")
print("=" * 70)


# V3 — Resume Writing Benchmark

## What We Tested
17 models writing a tailored ATS-optimized resume for **Boeing Defense GenAI Data Scientist** role using **Jeff Calderon's verified profile** (from SKILL.md).

## Scoring (100 pts)
- **ATS Keyword Coverage (30)**: JD keyword overlap — required, preferred, soft skills
- **Fact Fidelity (25)**: No invented credentials (employers, degrees, certs, clearance)
- **Section Structure (15)**: Header, Summary, Experience, Projects, Education, Certs
- **Bullet Quality (15)**: Action verbs, quantified metrics, employer relevance
- **JD Alignment (15)**: How well resume maps to specific Boeing JD phrases

## Key Finding
**ALL 17 models scored 25/25 on Fact Fidelity** — zero hallucinated credentials. No model invented a Master's degree, claimed clearance, or fabricated employers.

In [ ]:
# ============================================================================
# V3 RESUME WRITING BENCHMARK — 17 Models Combined
# ============================================================================
import json, pandas as pd, matplotlib.pyplot as plt, numpy as np

with open('/home/skystarved/dev_workspaces/results/v3_resume_writing_results.json', 'r') as f:
    v3 = json.load(f)

data = []
for e in v3:
    data.append({
        'model': e['model'],
        'score': e['score'],
        'ats': float(e.get('ats', e.get('ats_score', 0))),
        'fidelity': float(e.get('fidelity', e.get('fidelity_score', 0))),
        'structure': float(e.get('structure', e.get('section_score', 0))),
        'bullets': float(e.get('bullets', e.get('bullet_score', 0))),
        'alignment': float(e.get('alignment', e.get('alignment_score', 0))),
        'time': float(e.get('latency', 0)),
        'tokens': int(e.get('tokens', 0)),
        'cost': float(e.get('cost', 0)),
        'words': int(float(e.get('words', e.get('word_count', 0)))),
        'forbidden': int(float(e.get('forbidden', e.get('forbidden_hits', 0)))),
        'bullet_count': int(float(e.get('bullet_count', 0))),
        'metric_bullets': int(float(e.get('metric_bullets', 0))),
    })

bench_df = pd.DataFrame(data).sort_values('score', ascending=False)

print("=" * 85)
print("V3 RESUME WRITING BENCHMARK — Combined Results (17 models)")
print("=" * 85)
print(f"JD: Boeing Defense Data Scientist (GenAI)")
print(f"Scoring: ATS(30) + Fidelity(25) + Structure(15) + Bullets(15) + Alignment(15) = 100")
print(f"ALL models: 25/25 fact fidelity — zero hallucinated credentials\n")

print(f"{'#':<3} {'Model':<26} {'Score':<6} {'ATS':<6} {'Fid':<5} {'Sec':<5} {'Bul':<5} {'Ali':<5} {'Time':<7} {'Cost':<8}")
print("-" * 95)
for i, (_, r) in enumerate(bench_df.iterrows(), 1):
    print(f"{i:<3} {r['model']:<26} {r['score']:.1f}   {r['ats']:.1f}   {r['fidelity']:.0f}   {r['structure']:.1f}   {r['bullets']:<5.0f} {r['alignment']:.1f}   {r['time']:.0f}s{'':<3} ${r['cost']:.5f}")

# ── 6-PLOT DASHBOARD ──
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 1) Overall Score
ax = axes[0, 0]; d = bench_df.sort_values('score')
bars = ax.barh(d['model'], d['score'], color=plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(d))))
ax.set_title('Overall Resume Writing Score'); ax.set_xlabel('Score (100 pts)')
for b, s in zip(bars, d['score']): ax.text(s + 0.1, b.get_y() + b.get_height()/2, f'{s:.1f}', va='center', fontsize=7)

# 2) ATS Keyword Coverage
ax = axes[0, 1]; d = bench_df.sort_values('ats')
bars = ax.barh(d['model'], d['ats'], color=plt.cm.Blues(np.linspace(0.3, 0.9, len(d))))
ax.set_title('ATS Keyword Coverage (30 pts)'); ax.set_xlabel('ATS Score')
for b, s in zip(bars, d['ats']): ax.text(s + 0.1, b.get_y() + b.get_height()/2, f'{s:.1f}', va='center', fontsize=7)

# 3) Speed
ax = axes[0, 2]; d = bench_df.sort_values('time')
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(d)))[::-1]
bars = ax.barh(d['model'], d['time'], color=colors)
ax.set_title('Speed to Generate'); ax.set_xlabel('Time (seconds)')
for b, t in zip(bars, d['time']): ax.text(t + 1, b.get_y() + b.get_height()/2, f'{t:.0f}s', va='center', fontsize=7)

# 4) Stacked breakdown
ax = axes[1, 0]; d = bench_df.sort_values('score')
ax.barh(d['model'], d['ats'], label='ATS(30)', color='#3498db')
ax.barh(d['model'], d['fidelity'], left=d['ats'], label='Fidelity(25)', color='#2ecc71')
ax.barh(d['model'], d['structure'], left=d['ats'] + d['fidelity'], label='Structure(15)', color='#f39c12')
ax.barh(d['model'], d['bullets'], left=d['ats'] + d['fidelity'] + d['structure'], label='Bullets(15)', color='#e74c3c')
ax.barh(d['model'], d['alignment'], left=d['ats'] + d['fidelity'] + d['structure'] + d['bullets'], label='Align(15)', color='#9b59b6')
ax.set_title('Score Breakdown'); ax.set_xlabel('Points'); ax.legend(loc='lower right', fontsize=7)

# 5) Cost vs Score with Color-Coded Models and Legend
ax = axes[1, 1]

# Create a color map for each model using tab20 colormap
unique_models = bench_df['model'].tolist()
colors = plt.cm.tab20(np.linspace(0, 1, len(unique_models)))
model_color_map = {model: color for model, color in zip(unique_models, colors)}

# Plot each model with its own color
for model in unique_models:
    model_data = bench_df[bench_df['model'] == model]
    ax.scatter(
        model_data['cost'] * 1000, 
        model_data['score'], 
        s=model_data['time'] * 2, 
        alpha=0.8, 
        c=[model_color_map[model]], 
        label=model[:25],  # Truncate for legend
        edgecolors='black', 
        linewidth=0.5
    )

ax.set_xlabel('Cost (millicents)'); ax.set_ylabel('Score'); ax.set_title('Cost vs Score (color = model, size = time)')

# Add legend with model names (show all models in 2 columns)
handles, labels = ax.get_legend_handles_labels()
# Place legend outside the plot region on the right side
# bbox_to_anchor uses axes fraction coordinates; using (1.02, 0.5) attaches it
# just beyond the right edge, centered vertically.
ax.legend(
    handles,
    labels,
    loc='center left',
    fontsize=6,
    ncol=1,
    framealpha=0.9,
    bbox_to_anchor=(1.02, 0.5),
    borderaxespad=0.0,
)

# 6) Bullet Quality
ax = axes[1, 2]
ax.scatter(bench_df['bullet_count'], bench_df['metric_bullets'], s=bench_df['score'] * 3, alpha=0.7, c=bench_df['score'], cmap='RdYlGn', edgecolors='black', linewidth=0.5)
ax.set_xlabel('Total Bullets'); ax.set_ylabel('Bullets with Metrics'); ax.set_title('Bullet Quality (size = score)')
for _, r in bench_df.iterrows(): ax.annotate(r['model'][:14], (r['bullet_count'], r['metric_bullets']), xytext=(3, 2), textcoords='offset points', fontsize=7)
ax.plot([0, max(bench_df['bullet_count'])], [0, max(bench_df['bullet_count'])], 'k--', alpha=0.2)

plt.suptitle('V3 Resume Writing: 17 LLMs Tailoring Resumes for Boeing GenAI Role', fontsize=14, fontweight='bold', y=1.01)
note_source(SRC_BENCH)
plt.tight_layout(); plt.show()

# ── HALLUCINATION AUDIT ──
print("\nHALLUCINATION AUDIT — All models pass (0 invented credentials)")
for _, r in bench_df.iterrows():
    flag = f"⚠️ {r['forbidden']} flags" if r['forbidden'] > 0 else "✅ CLEAN"
    print(f"  {r['model']:<28} {flag}")

# ── RECOMMENDATIONS ──
print("\n" + "=" * 70)
print("RECOMMENDATIONS (17 models)")
print("=" * 70)
best = bench_df.iloc[0]; fastest = bench_df.loc[bench_df['time'].idxmin()]; cheapest = bench_df.loc[bench_df['cost'].idxmin()]
bench_df['value'] = bench_df['score'] / bench_df['cost'].replace(0, 1e-6)
best_val = bench_df.loc[bench_df['value'].idxmax()]
print(f"  🏆 Best Overall:    {best['model']} ({best['score']:.1f}/100)")
print(f"  ⚡ Fastest:         {fastest['model']} ({fastest['time']:.0f}s)")
print(f"  💰 Cheapest:        {cheapest['model']} (${cheapest['cost']:.4f})")
print(f"  📊 Best Value:      {best_val['model']} ({best_val['value']:.0f} score/dollar)")
print(f"  🔒 All models:      25/25 fact fidelity — zero hallucination")
print("=" * 70)
